In [ ]:
# ============================================================================
# NetworKit-based PageRank Simulation - EXPERT-LED FINEWEB (WWW) HOST GRAPH
# Includes: FAST C++ Graph Copy, Metric Triad, Per-Run Saving, & Checkpoints
# ============================================================================

!pip install networkit pandas numpy matplotlib seaborn tqdm
import pandas as pd
import numpy as np
import random
import networkit as nk
import time
import gc
import pickle
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm

# === BOOSTING CONFIGURATION ===
NUM_BOOSTING_ROUNDS = 20
BRIDGINGS_PER_ROUND = 25
TOTAL_SIMULATIONS = NUM_BOOSTING_ROUNDS * BRIDGINGS_PER_ROUND

# === THREE CONNECTION RANGES ===
CONNECTION_RANGES = [
    (5, 35, "Range_5-35"),
    (35, 65, "Range_35-65"),
    (65, 95, "Range_65-95"),
]

# === THRESHOLD CONFIGURATION ===
NEUTRAL_THRESHOLD = 0.025  # ±0.025% considered neutral

# === SIMULATION PARAMETERS ===
PAGERANK_TOLERANCE = 1e-6
PAGERANK_DAMPING = 0.80

# === FILE PATHS ===
FINEWEB_PATH = "/content/drive/MyDrive/WebKnoGraph/results/fineweb_500k_pages.csv"
BASELINE_PATH = "/content/drive/MyDrive/WebKnoGraph/results/link_graph_edges.csv"

COMPARISON_FOLDERS = [
    (
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/random_batches/",
        "Random Candidates",
    ),
    (
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/high_batches/",
        "High Candidates",
    ),
    (
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/folder_batches/",
        "Folder Candidates",
    ),
    (
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/mixed_batches/",
        "Mixed Candidates",
    ),
    (
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/low_batches/",
        "Low Candidates",
    ),
]

# === GLOBALS FOR CACHING ===
_cached_www_graph = None
_cached_www_nodes = None


# ============================================================================
# CHECKPOINT SYSTEM
# ============================================================================
class CheckpointManager:
    """Manages checkpoint saving and loading for resumable simulations"""

    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True, parents=True)
        self.checkpoint_file = (
            self.checkpoint_dir / "simulation_checkpoint_expert_www.pkl"
        )
        self.completed_combinations = set()
        self.load_checkpoint()

    def load_checkpoint(self):
        if self.checkpoint_file.exists():
            try:
                with open(self.checkpoint_file, "rb") as f:
                    data = pickle.load(f)
                    self.completed_combinations = data.get("completed", set())
                print(
                    f"✓ Loaded checkpoint: {len(self.completed_combinations)} combinations already completed"
                )
            except Exception as e:
                print(f"⚠ Could not load checkpoint: {e}")
                self.completed_combinations = set()

    def save_checkpoint(self, strategy_name, range_name, min_conn, max_conn):
        combination = (strategy_name, range_name, min_conn, max_conn)
        self.completed_combinations.add(combination)
        checkpoint_data = {
            "completed": self.completed_combinations,
            "last_updated": datetime.now().isoformat(),
        }
        try:
            with open(self.checkpoint_file, "wb") as f:
                pickle.dump(checkpoint_data, f)
            print(f"  ✓ Checkpoint saved ({len(self.completed_combinations)} total)")
        except Exception as e:
            print(f"  ⚠ Could not save checkpoint: {e}")

    def is_completed(self, strategy_name, range_name, min_conn, max_conn):
        return (
            strategy_name,
            range_name,
            min_conn,
            max_conn,
        ) in self.completed_combinations


def mount_google_drive():
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        print("✓ Google Drive mounted successfully!")
        return True
    except:
        print("⚠ Not in Colab - skipping drive mount")
        return False


# ============================================================================
# GRAPH LOADING & PROCESSING
# ============================================================================
def load_graph_from_csv_networkit(file_path):
    """Load directed graph from CSV with FROM/TO columns"""
    try:
        df = pd.read_csv(file_path, usecols=["FROM", "TO"])
        df = df.dropna()
        df["FROM"] = df["FROM"].astype(str)
        df["TO"] = df["TO"].astype(str)
    except Exception as e:
        print(f"✗ Error loading {file_path}: {str(e)}")
        return None, None, None

    from_urls = df["FROM"].values
    to_urls = df["TO"].values

    if len(from_urls) == 0:
        return None, None, None

    all_urls = np.unique(np.concatenate([from_urls, to_urls]))
    url_to_idx = {url: i for i, url in enumerate(all_urls)}

    g = nk.Graph(n=len(all_urls), weighted=False, directed=True)
    for src_url, tgt_url in zip(from_urls, to_urls):
        g.addEdge(url_to_idx[src_url], url_to_idx[tgt_url])

    return g, all_urls, url_to_idx


def get_cached_www_graph():
    """Load and cache the Master FineWeb graph exactly once to save massive amounts of time"""
    global _cached_www_graph, _cached_www_nodes
    if _cached_www_graph is None:
        print(f"\n  Loading Master FineWeb Graph from {Path(FINEWEB_PATH).name}...")
        g, nodes, _ = load_graph_from_csv_networkit(FINEWEB_PATH)
        if g is None:
            raise RuntimeError(f"Failed to load FineWeb graph from {FINEWEB_PATH}")
        _cached_www_graph = g
        _cached_www_nodes = nodes
        print(
            f"  ✓ Cached: {_cached_www_graph.numberOfNodes():,} nodes, {_cached_www_graph.numberOfEdges():,} edges"
        )

    # FAST C++ COPY: Takes milliseconds instead of minutes!
    new_graph = nk.Graph(_cached_www_graph)

    return new_graph, len(_cached_www_nodes)


def process_configuration_networkit(
    www_graph,
    total_www_nodes,
    kalicube_edges,
    kalicube_nodes,
    min_connections,
    max_connections,
):
    """Merge Kalicube graph with WWW Graph and calculate PageRank"""
    kalicube_offset = total_www_nodes
    n_kalicube = len(kalicube_nodes)

    for _ in range(n_kalicube):
        www_graph.addNode()

    if kalicube_edges:
        for src, tgt in kalicube_edges:
            www_graph.addEdge(src + kalicube_offset, tgt + kalicube_offset)

    n_connections = np.random.randint(min_connections, max_connections + 1)
    n_www_sample = min(n_connections, total_www_nodes)
    n_kalicube_sample = min(n_connections, n_kalicube)

    www_nodes_sample = np.random.choice(
        total_www_nodes, size=n_www_sample, replace=False
    )
    kalicube_indices = np.random.choice(
        n_kalicube, size=n_kalicube_sample, replace=False
    )

    for www_node_id, kalicube_idx in zip(www_nodes_sample, kalicube_indices):
        www_graph.addEdge(www_node_id, kalicube_idx + kalicube_offset)

    pagerank_algo = nk.centrality.PageRank(
        www_graph, damp=PAGERANK_DAMPING, tol=PAGERANK_TOLERANCE
    )
    pagerank_algo.run()
    pagerank_scores = pagerank_algo.scores()

    pagerank_dict = {}
    for i, url in enumerate(kalicube_nodes):
        pagerank_dict[url] = pagerank_scores[i + kalicube_offset]

    return pagerank_dict


# ============================================================================
# SIMULATION & METRICS
# ============================================================================
def run_boosting_round(
    round_id, old_edges, new_edges, old_nodes, new_nodes, min_conn, max_conn
):
    """Run one boosting round and calculate run-level averages"""
    run_stats = []

    for bridging_id in range(BRIDGINGS_PER_ROUND):
        sim_id = round_id * BRIDGINGS_PER_ROUND + bridging_id
        sim_seed = 42 + sim_id

        np.random.seed(sim_seed)
        random.seed(sim_seed)

        # Get fresh copies of the WWW graph for Old and New states
        www_graph_old, total_www_nodes = get_cached_www_graph()
        www_graph_new, _ = get_cached_www_graph()

        pagerank_old = process_configuration_networkit(
            www_graph_old, total_www_nodes, old_edges, old_nodes, min_conn, max_conn
        )
        pagerank_new = process_configuration_networkit(
            www_graph_new, total_www_nodes, new_edges, new_nodes, min_conn, max_conn
        )

        common_urls = set(pagerank_old.keys()) & set(pagerank_new.keys())
        if not common_urls:
            continue

        run_deltas = []
        u_r, d_r, n_r = 0, 0, 0

        for url in common_urls:
            old_val = pagerank_old[url]
            new_val = pagerank_new[url]
            delta_pct = ((new_val - old_val) / max(old_val, 1e-10)) * 100
            run_deltas.append(delta_pct)

            if delta_pct > NEUTRAL_THRESHOLD:
                u_r += 1
            elif delta_pct < -NEUTRAL_THRESHOLD:
                d_r += 1
            else:
                n_r += 1

        mu_r = np.mean(run_deltas) if run_deltas else 0
        run_stats.append(
            {
                "run_id": sim_id + 1,
                "mu_r": mu_r,
                "u_r": u_r,
                "d_r": d_r,
                "n_r": n_r,
                "total_pages": len(common_urls),
            }
        )

        del www_graph_old, www_graph_new, pagerank_old, pagerank_new
        gc.collect()

    return run_stats


def run_boosted_comparison(
    baseline_data,
    comparison_file,
    min_conn,
    max_conn,
    range_name,
    main_output_folder,
    strategy_name,
):
    """Run 500 simulations for a single file and apply the WebKnoGraph Metric Triad"""
    comparison_name = comparison_file.stem
    print(f"  Processing: {comparison_name} @ {range_name}...")

    start_time = time.time()

    g_old, nodes_old, _ = baseline_data
    g_new, nodes_new, _ = load_graph_from_csv_networkit(comparison_file)

    if g_new is None:
        print(f"✗ Failed")
        return None

    old_edges = [(u, v) for u, v in g_old.iterEdges()]
    new_edges = [(u, v) for u, v in g_new.iterEdges()]
    old_nodes = nodes_old
    new_nodes = nodes_new

    k = max(len(new_edges) - len(old_edges), 1)

    del g_new
    gc.collect()

    all_run_stats = []

    # Progress bar for the boosting rounds!
    for round_id in tqdm(
        range(NUM_BOOSTING_ROUNDS), desc=f"Simulating {comparison_name}", leave=False
    ):
        stats = run_boosting_round(
            round_id, old_edges, new_edges, old_nodes, new_nodes, min_conn, max_conn
        )
        all_run_stats.extend(stats)

    if not all_run_stats:
        print("✗ No results")
        return None

    # Save Per-Run Data (Volatility tracking)
    safe_strategy_name = strategy_name.replace(" ", "_")
    runs_dir = main_output_folder / "per_run_data" / safe_strategy_name / range_name
    runs_dir.mkdir(exist_ok=True, parents=True)

    run_df = pd.DataFrame(all_run_stats)
    run_df.insert(0, "Comparison_File", comparison_name)
    run_df.insert(1, "Links_Inserted_k", k)
    run_df["Per_Run_Authority_Yield"] = run_df["mu_r"] / k

    runs_file = runs_dir / f"{comparison_name}_runs.csv"
    run_df.to_csv(runs_file, index=False)

    # Summarize Run-Level metrics into WebKnoGraph Triad
    mu_array = np.array([s["mu_r"] for s in all_run_stats])
    u_array = np.array([s["u_r"] for s in all_run_stats])
    d_array = np.array([s["d_r"] for s in all_run_stats])
    n_array = np.array([s["n_r"] for s in all_run_stats])

    mean_delta_pct = np.mean(mu_array)
    std_delta = np.std(mu_array)

    avg_pages_up = np.mean(u_array)
    avg_pages_down = np.mean(d_array)
    avg_pages_neutral = np.mean(n_array)

    # THE METRIC TRIAD
    authority_yield = mean_delta_pct / k
    authority_volatility = std_delta / k
    down_up_ratio = avg_pages_down / max(avg_pages_up, 1.0)

    duration = time.time() - start_time
    print(f"\r  ✓ Yield: {authority_yield:+.4f}/link (k={k}) [{duration:.1f}s]")

    return {
        "name": comparison_name,
        "range_name": range_name,
        "min_connections": min_conn,
        "max_connections": max_conn,
        "duration": duration,
        "k_inserted": k,
        "mean_delta_pct": mean_delta_pct,
        "std_delta": std_delta,
        "authority_yield": authority_yield,
        "authority_volatility": authority_volatility,
        "avg_pages_up": avg_pages_up,
        "avg_pages_down": avg_pages_down,
        "avg_pages_neutral": avg_pages_neutral,
        "down_up_ratio": down_up_ratio,
        "total_pages": all_run_stats[0]["total_pages"],
    }


def create_strategy_summary(all_results, output_folder, strategy_name, range_name):
    print(f"\n{'=' * 70}")
    print(f"STRATEGY SUMMARY: {strategy_name} @ {range_name}")
    print(f"{'=' * 70}")

    data = []
    for r in all_results:
        if r:
            data.append(
                {
                    "Comparison": r["name"],
                    "Range": r["range_name"],
                    "Links_Inserted_k": r["k_inserted"],
                    "Mean_Delta_%": r["mean_delta_pct"],
                    "Std_Delta_%": r["std_delta"],
                    "Authority_Yield_AY": r["authority_yield"],
                    "Authority_Volatility_AV": r["authority_volatility"],
                    "Down_Up_Ratio_DUR": r["down_up_ratio"],
                    "Avg_Pages_Up": r["avg_pages_up"],
                    "Avg_Pages_Down": r["avg_pages_down"],
                    "Avg_Pages_Neutral": r["avg_pages_neutral"],
                }
            )

    if not data:
        return None

    df = pd.DataFrame(data)
    df = df.sort_values("Authority_Yield_AY", ascending=False)

    formatted_strategy = "_".join(word.capitalize() for word in strategy_name.split())
    summary_path = output_folder / f"{range_name}_{formatted_strategy}.csv"
    df.to_csv(summary_path, index=False)

    print("\nRankings by Authority Yield (AY):")
    for idx, row in df.iterrows():
        symbol = "↑" if row["Authority_Yield_AY"] > 0 else "↓"
        print(
            f"  {symbol} {row['Comparison']}: AY={row['Authority_Yield_AY']:+.4f} | AV={row['Authority_Volatility_AV']:.4f} | DUR={row['Down_Up_Ratio_DUR']:.2f}"
        )

    return {
        "strategy_name": strategy_name,
        "range_name": range_name,
        "min_connections": all_results[0]["min_connections"],
        "max_connections": all_results[0]["max_connections"],
        "avg_k": df["Links_Inserted_k"].mean(),
        "avg_ay": df["Authority_Yield_AY"].mean(),
        "avg_av": df["Authority_Volatility_AV"].mean(),
        "avg_dur": df["Down_Up_Ratio_DUR"].mean(),
        "avg_up": df["Avg_Pages_Up"].mean(),
        "avg_down": df["Avg_Pages_Down"].mean(),
        "num_comparisons": len(data),
    }


def update_overall_tracker(overall_tracker_path, strategy_result):
    if overall_tracker_path.exists():
        tracker_df = pd.read_csv(overall_tracker_path)
        existing_data = tracker_df.to_dict("records")
    else:
        existing_data = []

    existing_data.append(
        {
            "Strategy": strategy_result["strategy_name"],
            "Range": strategy_result["range_name"],
            "Min_Connections": strategy_result["min_connections"],
            "Max_Connections": strategy_result["max_connections"],
            "Avg_Links_Inserted_k": strategy_result["avg_k"],
            "Overall_Avg_AY": strategy_result["avg_ay"],
            "Overall_Avg_AV": strategy_result["avg_av"],
            "Overall_Avg_DUR": strategy_result["avg_dur"],
            "Overall_Avg_Pages_Up": strategy_result["avg_up"],
            "Overall_Avg_Pages_Down": strategy_result["avg_down"],
            "Num_Comparisons": strategy_result["num_comparisons"],
        }
    )

    tracker_df = pd.DataFrame(existing_data)
    tracker_df.to_csv(overall_tracker_path, index=False)


# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("NetworKit PageRank Simulation: EXPERT-LED FINEWEB (WWW) HOST ENVIRONMENT")
    print("FAST C++ Graph Copy, WebKnoGraph Metrics, & PER-RUN saving")
    print("=" * 70)

    mount_google_drive()

    baseline_path = Path(BASELINE_PATH)
    if not baseline_path.exists():
        print(f"\n✗ Baseline not found: {BASELINE_PATH}")
        exit(1)

    print("\nLoading baseline Kalicube graph...")
    baseline_data = load_graph_from_csv_networkit(baseline_path)
    print(
        f"✓ Loaded: {baseline_data[0].numberOfNodes():,} nodes, {baseline_data[0].numberOfEdges():,} edges"
    )

    # Outputs cleanly to the expert_led folder without overwriting BA runs
    main_output_folder = Path(
        "/content/drive/MyDrive/WebKnoGraph/results/expert_led/fineweb_results_expert"
    )
    main_output_folder.mkdir(exist_ok=True, parents=True)

    checkpoint_manager = CheckpointManager(main_output_folder / "checkpoints_www")
    overall_tracker_path = (
        main_output_folder / "OVERALL_AVERAGES_TRACKER_EXPERT_WWW.csv"
    )

    all_strategy_results = []
    run_count = 0
    total_runs = len(COMPARISON_FOLDERS) * len(CONNECTION_RANGES)

    for min_conn, max_conn, range_name in CONNECTION_RANGES:
        print(f"\n\n{'█' * 70}")
        print(f"CONNECTION RANGE: {range_name} ({min_conn}-{max_conn})")
        print(f"{'█' * 70}")

        for folder_path, strategy_name in COMPARISON_FOLDERS:
            run_count += 1

            print(f"\n[RUN {run_count}/{total_runs}]")

            if checkpoint_manager.is_completed(
                strategy_name, range_name, min_conn, max_conn
            ):
                print(f"⏭ SKIPPING (already completed): {strategy_name} @ {range_name}")
                continue

            comparison_folder = Path(folder_path)
            comparison_files = list(comparison_folder.glob("*.csv"))

            if not comparison_files:
                print(f"✗ No CSV files in: {comparison_folder}")
                continue

            all_results = []
            for comp_file in comparison_files:
                result = run_boosted_comparison(
                    baseline_data,
                    comp_file,
                    min_conn,
                    max_conn,
                    range_name,
                    main_output_folder,
                    strategy_name,
                )
                if result:
                    all_results.append(result)

            strategy_result = create_strategy_summary(
                all_results, main_output_folder, strategy_name, range_name
            )

            if strategy_result:
                update_overall_tracker(overall_tracker_path, strategy_result)
                checkpoint_manager.save_checkpoint(
                    strategy_name, range_name, min_conn, max_conn
                )
                all_strategy_results.append(strategy_result)

    print(f"\n{'=' * 70}")
    print("✓ ALL EXPERT-LED FINEWEB STRATEGY-RANGE COMBINATIONS PROCESSED")
    print(f"{'=' * 70}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 43.2 MB/s eta 0:00:00


NetworKit PageRank Simulation: EXPERT-LED FINEWEB (WWW) HOST ENVIRONMENT
FAST C++ Graph Copy, WebKnoGraph Metrics, & PER-RUN saving
Mounted at /content/drive
✓ Google Drive mounted successfully!

Loading baseline Kalicube graph...
✓ Loaded: 1,841 nodes, 122,066 edges


██████████████████████████████████████████████████████████████████████
CONNECTION RANGE: Range_5-35 (5-35)
██████████████████████████████████████████████████████████████████████

[RUN 1/15]
  Processing: 240_random_updated_link_graph_1 @ Range_5-35...


Simulating 240_random_updated_link_graph_1:   0%|          | 0/20 [00:00<?, ?it/s]


  Loading Master FineWeb Graph from fineweb_500k_pages.csv...
  ✓ Cached: 224,242 nodes, 500,000 edges


  ✓ Yield: +0.0091/link (k=240) [550.7s]
  Processing: 240_random_updated_link_graph_2 @ Range_5-35...


...

  ✓ Yield: +0.0022/link (k=240) [549.0s]

STRATEGY SUMMARY: Folder Candidates @ Range_5-35

Rankings by Authority Yield (AY):
  ↑ 240_folder_updated_link_graph_1: AY=+0.0039 | AV=0.0017 | DUR=2.03
  ↑ 240_folder_updated_link_graph_2: AY=+0.0022 | AV=0.0017 | DUR=1.91
  ✓ Checkpoint saved (3 total)

[RUN 4/15]
  Processing: 240_mixed_updated_link_graph_2 @ Range_5-35...


  ✓ Yield: +0.0013/link (k=240) [549.3s]
  Processing: 240_mixed_updated_link_graph_1 @ Range_5-35...


  ✓ Yield: +0.0109/link (k=240) [545.8s]

STRATEGY SUMMARY: Mixed Candidates @ Range_5-35

Rankings by Authority Yield (AY):
  ↑ 240_mixed_updated_link_graph_1: AY=+0.0109 | AV=0.0017 | DUR=3.46
  ↑ 240_mixed_updated_link_graph_2: AY=+0.0013 | AV=0.0017 | DUR=1.43
  ✓ Checkpoint saved (4 total)

[RUN 5/15]
  Processing: 240_low_updated_link_graph_2 @ Range_5-35...


  ✓ Yield: +0.0018/link (k=240) [559.1s]
  Processing: 240_low_updated_link_graph_1 @ Range_5-35...


Simulating 240_low_updated_link_graph_1:  40%|████      | 8/20 [03:47<05:42, 28.57s/it]